In [2]:
!pip install ultralytics

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="hkTkjFCZ5Ul7fGujCKmA")
project = rf.workspace("intellifone").project("damage-detection-d0coe")
version = project.version(7)
dataset = version.download("yolov11")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 118.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Damage-detection-7 in yolov11:: 100%|██████████| 6441/6441 [00:00<00:00, 7198.62it/s]


In [7]:
import os, yaml, glob
from collections import Counter

dataset_dir = "/content/Damage-detection-7"
keep_classes = ["crack", "dot", "line"]

data_yaml = os.path.join(dataset_dir, "data.yaml")
with open(data_yaml, "r") as f:
    data = yaml.safe_load(f)

class_names = data["names"]
missing = [c for c in keep_classes if c not in class_names]
if missing:
    raise ValueError(f"Missing classes in dataset: {missing}")

keep_ids = [class_names.index(c) for c in keep_classes]
print(f"Found classes: {class_names}")
print(f"Keeping class IDs: {keep_ids} -> {keep_classes}")

for split in ["train", "valid", "test"]:
    label_dir = os.path.join(dataset_dir, split, "labels")
    if not os.path.exists(label_dir):
        continue

    for file in os.listdir(label_dir):
        path = os.path.join(label_dir, file)
        with open(path, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            cls_id = int(parts[0])
            if cls_id in keep_ids:
                parts[0] = str(keep_ids.index(cls_id))
                new_lines.append(" ".join(parts) + "\n")

        if new_lines:
            with open(path, "w") as f:
                f.writelines(new_lines)
        else:
            os.remove(path)

data["names"] = keep_classes
data["nc"] = len(keep_classes)
with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f)

counts = Counter()
for split in ["train", "valid", "test"]:
    label_dir = os.path.join(dataset_dir, split, "labels")
    if not os.path.exists(label_dir):
        continue
    for label_file in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(label_file, "r") as f:
            for line in f:
                parts = line.split()
                if parts:
                    counts[int(parts[0])] += 1

print("Label count after filtering:")
for i, name in enumerate(keep_classes):
    print(f"{i} ({name}): {counts.get(i, 0)}")
print(f"Total labeled objects: {sum(counts.values())}")


Found classes: ['crack', 'dot', 'line']
Keeping class IDs: [0, 1, 2] -> ['crack', 'dot', 'line']
Label count after filtering:
0 (crack): 5916
1 (dot): 1116
2 (line): 1977
Total labeled objects: 9009


In [8]:
!yolo task=segment mode=train \
    model=yolo11n-seg.pt \
    data=/content/Damage-detection-7/data.yaml \
    epochs=50 \
    imgsz=640 \
    batch=16 \
    device=0 \
    project="/content/drive/MyDrive/yolo_runs" \
    name="best_model" \
    cache=True \
    workers=4


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Damage-detection-7/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.

In [ ]:
!rm /content/Damage-detection-6/valid/labels.cache
!rm /content/Damage-detection-6/test/labels.cache


rm: cannot remove '/content/Damage-detection-6/test/labels.cache': No such file or directory


In [10]:
!yolo task=segment mode=val \
    model="/content/drive/MyDrive/yolo_runs/best_model/weights/best.pt" \
    data=/content/Damage-detection-7/data.yaml \
    split=test \
    device=0

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-seg summary (fused): 114 layers, 2,835,153 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 17.4±2.3 MB/s, size: 38.5 KB)
val: Scanning /content/Damage-detection-7/test/labels... 154 images, 25 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 179/179 424.1it/s 0.4s
val: New cache created: /content/Damage-detection-7/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.2it/s 5.4s
                   all        179        293      0.696      0.575       0.64      0.424      0.648      0.533      0.559      0.281
                 crack         48         92      0.498      0.293       0.35      0.263      0.428       0.25      0.238      0.144
                   dot         55         67      0.804      0.776      0.862      0.538      0.778      0

In [11]:
!yolo val \
    model=/content/drive/MyDrive/yolo_runs/best_model/weights/best.pt \
    data=/content/Damage-detection-7/data.yaml \
    task=segment \
    device=0


Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-seg summary (fused): 114 layers, 2,835,153 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 123.4±243.8 MB/s, size: 33.8 KB)
val: Scanning /content/Damage-detection-7/valid/labels.cache... 144 images, 35 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 179/179 21.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 2.2it/s 5.4s
                   all        179        315      0.668      0.521      0.572      0.387      0.637       0.45      0.493      0.278
                 crack         49        130      0.635      0.227      0.299       0.18      0.507      0.169      0.186      0.119
                   dot         52         68      0.734      0.676      0.727      0.485      0.734      0.618      0.681      0.435
                  line         59

In [ ]:
!yolo predict \
    model="/content/drive/MyDrive/yolo_runs/best_model/weights/best.pt" \
    source="/content/mypic.jpeg" \
    save=True \
    save_txt=True

Ultralytics 8.3.207 🚀 Python-3.12.11 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11s-seg summary (fused): 113 layers, 10,067,977 parameters, 0 gradients, 32.8 GFLOPs

image 1/1 /content/mypic.jpeg: 640x384 3 lines, 491.3ms
Speed: 3.7ms preprocess, 491.3ms inference, 11.1ms postprocess per image at shape (1, 3, 640, 384)
Results saved to /content/runs/segment/predict5
1 label saved to /content/runs/segment/predict5/labels
💡 Learn more at https://docs.ultralytics.com/modes/predict


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo_runs/best_model/weights/best.pt")
results = model.predict("/content/mypic.jpeg")

print(results[0].orig_shape)   # (H, W) of original image
print(results[0].masks.data.shape if results[0].masks else "No masks")



image 1/1 /content/mypic.jpeg: 640x384 3 lines, 734.5ms
Speed: 14.8ms preprocess, 734.5ms inference, 40.3ms postprocess per image at shape (1, 3, 640, 384)
(1280, 720)
torch.Size([3, 640, 384])
